# Bandit Auction — Discrete Multi-Good

Two buyers, two goods, discrete types {1, 2, 3} per good (9 joint types per buyer), horizon T=5.
No closed-form benchmark for multi-good discrete setting.

Acceptance-based training with **eval IC** (oracle re-opt restarts) gating checkpoints.
Set `RUN_ENV = 'colab'` or `'local'`.

In [ ]:
RUN_ENV = 'colab'   # 'colab' | 'local'
LOCAL_PROJECT_DIR = None
# LOCAL_PROJECT_DIR = r'E:\Research\code\Dynamic Mechanism Design\Bandit Auction'

In [ ]:
import os, sys, subprocess

if RUN_ENV == 'colab':
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y',
        'jax', 'jaxlib', 'jax-cuda12-plugin', 'jax-cuda12-pjrt'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-qU',
        'jax[cuda12]', 'dm-haiku', 'optax', 'flax'])
    print('Install done. IMPORTANT: Runtime → Restart session, then re-run from this cell.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.chdir('/content/drive/MyDrive/My Research/Dynamic Mechanism Design/Bandit Auction')
        print(f'Working directory: {os.getcwd()}')
    except ImportError:
        print('(Drive mount skipped — restart first.)')
elif RUN_ENV == 'local':
    if LOCAL_PROJECT_DIR:
        os.chdir(LOCAL_PROJECT_DIR)
    print(f'Working directory: {os.getcwd()}')

In [ ]:
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pickle, json, os

from bandit_auction.core import (
    AuctionSimulator, DiscretePolicy, build_discrete_policy_net,
)
from bandit_auction.mechanisms.network import Mechanism, build_mechanism_network
from bandit_auction.train.binary_search import (
    train_with_checkpoints, evaluate_checkpoint_list, DEFAULT_BS_CFG,
)
from bandit_auction.train.evaluate import eval_mechanism_multikey, print_eval_report
from bandit_auction.envs.discrete_multi import create_discrete_multi_auction

print('JAX devices:', jax.devices())

## Configuration

In [ ]:
EXPERIMENT  = 'discrete_multi'
IS_DISCRETE = True
PROFIT_GOAL = 8.0    # no closed-form benchmark; set based on prior runs
SEED        = 42

cfg = {**DEFAULT_BS_CFG,
    # Training
    'lr_theta':           1e-4,
    'lr_w':               5e-3,
    'num_outer_epochs':   2000,
    'num_inner_steps':    500,
    'num_samples_train':  64,
    'alpha_base':         5.0,
    'beta_weight':        1.0,
    'pg_base':            PROFIT_GOAL,
    'print_iter':         50,
    # Checkpoint gating (training IC/IR)
    'checkpoint_thresh':  0.15,
    'checkpoint_interval': 50,
    # Eval IC (re-opt oracle)
    'ic_thresh':          0.05,
    'gap_thresh':         0.1,
    'ic_reopt_steps':     600,
    'ic_reopt_restarts':  3,
    'ic_reopt_lr':        5e-4,
    'ic_reopt_samples':   128,
    # Final eval
    'eval_num_keys':      32,
    'eval_num_samples':   256,
}
print('Config loaded. profit_goal =', PROFIT_GOAL)

## Build Environment and Networks

In [ ]:
auction = create_discrete_multi_auction(
    n_buyers=2, n_goods=2, horizon=5, discount=0.9,
    type_values_per_good=[1.0, 2.0, 3.0], allocation_cost=0.1, seed=SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}')
print(f'Type values per good: [1.0, 2.0, 3.0]  ({3**auction.num_goods} joint types per buyer)')

s0 = auction.get_initial_state(jax.random.PRNGKey(0))
print(f'Initial types: {s0.current_type}')

In [ ]:
mechanism_network = build_mechanism_network()
policy_network = build_discrete_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub, hidden=128,
)

_prng = hk.PRNGSequence(SEED)
_mnet = hk.without_apply_rng(hk.transform(mechanism_network))
_b0   = auction.get_initial_belief_state(s0)
_mp   = _mnet.init(next(_prng), s0, auction)
_pp   = policy_network.init(next(_prng), _b0, auction)
print(f'Mechanism params: {sum(p.size for p in jax.tree_util.tree_leaves(_mp)):,}')
print(f'Policy params:    {sum(p.size for p in jax.tree_util.tree_leaves(_pp)):,}')

## Test: Oracle Inner Loop (Maximize Cumulative Regret)

Run the inner optimization loop against a **fixed** mechanism to verify that the oracle converges and finds profitable deviations. This is a sanity check before full training.

- `TEST_MECHANISM_PARAMS = None` uses a **randomly initialized** mechanism.
- Set it to trained params (e.g. `best_mechanism_params` from a prior run) to measure how much regret a trained mechanism actually allows.

The regret curve should climb from near-0 and plateau. If it stays flat at 0 throughout, the oracle is not finding deviations — increase `ORACLE_TEST_STEPS` or check the policy initialization.

In [ ]:
# ---- Oracle test configuration ----
ORACLE_TEST_STEPS    = 500    # inner gradient-ascent steps (matches num_inner_steps)
ORACLE_TEST_RESTARTS = 3      # independent random restarts
ORACLE_TEST_LR       = cfg['lr_w']  # reuse training inner LR
ORACLE_TEST_SAMPLES  = 64     # MC samples per regret estimate
TEST_MECHANISM_PARAMS = None  # None → random init; or set to best_mechanism_params

# Change this seed to test a different random mechanism.
ORACLE_MECHANISM_SEED = 77

# ---- Build oracle networks ----
_oracle_mnet = hk.without_apply_rng(hk.transform(mechanism_network))
_oracle_pnet = build_discrete_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub, hidden=128,
)

# ---- Initialize test mechanism params ----
_oracle_prng = hk.PRNGSequence(ORACLE_MECHANISM_SEED)
_oracle_s0   = auction.get_initial_state(next(_oracle_prng))
if TEST_MECHANISM_PARAMS is None:
    _test_mparams = _oracle_mnet.init(next(_oracle_prng), _oracle_s0, auction)
    print(f'Using randomly initialized mechanism (ORACLE_MECHANISM_SEED={ORACLE_MECHANISM_SEED}).')
else:
    _test_mparams = TEST_MECHANISM_PARAMS
    print('Using provided mechanism params.')
_test_mechanism = Mechanism(_oracle_mnet, _test_mparams)

# ---- JIT-compiled oracle step ----
_oracle_opt = optax.adam(ORACLE_TEST_LR)

@jax.jit
def _oracle_step(policy_params, w_opt_state, key):
    _reg, _grad = jax.value_and_grad(
        lambda pp: AuctionSimulator.estimate_cumulative_regret(
            auction, _test_mechanism,
            DiscretePolicy(_oracle_pnet, pp),
            ORACLE_TEST_SAMPLES, key,
            auction.get_next_belief_state_discrete,   # discrete belief update
        )
    )(policy_params)
    _grad_asc = jax.tree_util.tree_map(lambda g: -g, _grad)
    _upd, w_opt_state = _oracle_opt.update(_grad_asc, w_opt_state, policy_params)
    policy_params = optax.apply_updates(policy_params, _upd)
    return float(_reg), policy_params, w_opt_state

# ---- Run oracle for each restart ----
_oracle_curves   = []
_oracle_best_reg = -float('inf')
_oracle_key_base = ORACLE_MECHANISM_SEED * 1_000_000

for _restart in range(ORACLE_TEST_RESTARTS):
    _b0  = auction.get_initial_belief_state(_oracle_s0)
    _pp  = _oracle_pnet.init(next(_oracle_prng), _b0, auction)
    _ws  = _oracle_opt.init(_pp)
    _curve = []
    print(f'Restart {_restart+1}/{ORACLE_TEST_RESTARTS}')
    for _step in range(ORACLE_TEST_STEPS):
        _key = jax.random.PRNGKey(_oracle_key_base + _restart * 100_000 + _step)
        _reg, _pp, _ws = _oracle_step(_pp, _ws, _key)
        _curve.append(_reg)
        if _step % 100 == 0:
            print(f'  step {_step:4d}: regret = {_reg:.6f}')
    _oracle_curves.append(_curve)
    _oracle_best_reg = max(_oracle_best_reg, max(_curve))
    print(f'  → restart best: {max(_curve):.6f}')

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for _r, _c in enumerate(_oracle_curves):
    axes[0].plot(_c, alpha=0.8, label=f'restart {_r+1}')
axes[0].axhline(0, color='red', linestyle='--', lw=1, label='0')
axes[0].set_xlabel('Inner Step'); axes[0].set_ylabel('Regret')
axes[0].set_title('Oracle Convergence (all restarts)'); axes[0].legend()
_best_c = _oracle_curves[max(range(len(_oracle_curves)), key=lambda r: max(_oracle_curves[r]))]
axes[1].plot(_best_c, color='tab:red')
axes[1].axhline(0, color='k', linestyle='--', lw=1)
axes[1].set_xlabel('Inner Step'); axes[1].set_ylabel('Regret')
axes[1].set_title('Best Restart')
plt.suptitle(f'{EXPERIMENT} — Oracle Inner Loop Test (fixed mechanism)', y=1.02)
plt.tight_layout(); plt.show()
print(f'\nBest regret across all restarts: {_oracle_best_reg:.6f}')
print('(If this stays near 0, the oracle cannot find deviations — '
      'mechanism may already be approximately IC, or oracle needs more steps/LR.)')

## Training with Checkpoint Recording

`train_with_checkpoints` runs nested GD and records every checkpoint where training IC ≤ `checkpoint_thresh` and training IR ≤ `checkpoint_thresh`. The final epoch is always included as a candidate.

**Note:** Checkpoints are gated on *training* losses (cheap). The next cell evaluates each checkpoint with the tighter *eval* IC criterion (oracle re-opt restarts).

In [ ]:
checkpoints, logs = train_with_checkpoints(
    auction=auction,
    mechanism_network=mechanism_network,
    policy_network=policy_network,
    profit_goal=PROFIT_GOAL,
    cfg=cfg,
    is_discrete=IS_DISCRETE,
    seed=SEED,
    verbose=True,
)
print(f'\nTraining complete. Candidate checkpoints: {len(checkpoints)}')

## Checkpoint Evaluation — Eval IC with Oracle Re-opt

For each candidate checkpoint, the oracle is re-optimized from scratch (`ic_reopt_restarts` × `ic_reopt_steps` gradient ascent steps). A checkpoint **PASS**es if:
- `max(IC_reopt, IC_eval) ≤ ic_thresh` (= 0.05)
- `profit_goal − revenue_eval ≤ gap_thresh` (= 0.1)

Checkpoints are sorted: feasible first, then by descending revenue.

In [ ]:
eval_results = evaluate_checkpoint_list(
    auction=auction,
    mechanism_network=mechanism_network,
    policy_network=policy_network,
    checkpoints=checkpoints,
    profit_goal=PROFIT_GOAL,
    cfg=cfg,
    is_discrete=IS_DISCRETE,
    seed=SEED,
    verbose=True,
)

feasible = [r for r in eval_results if r['feasible']]
print(f'\n{len(feasible)} / {len(eval_results)} checkpoints PASS eval IC.')
if feasible:
    best = feasible[0]
    best_mechanism_params = best['mechanism_params']
    best_policy_params    = best['policy_params']
    print(f'Best: epoch={best["epoch"]}  rev={best["revenue_eval"]:.4f}  '
          f'IC_reopt={best["regret_reopt"]:.4f}  IC_eval={best["regret_eval"]:.4f}')
else:
    print('No feasible checkpoint. Using checkpoint with lowest eval IC.')
    best = min(eval_results, key=lambda r: r['regret_eval'])
    best_mechanism_params = best['mechanism_params']
    best_policy_params    = best['policy_params']
    print(f'Lowest IC: epoch={best["epoch"]}  IC_eval={best["regret_eval"]:.4f}')

## Checkpoint Summary Table

In [ ]:
print(f'\n{"="*80}')
print(f'  {EXPERIMENT} — Checkpoint Evaluation Summary  (profit_goal={PROFIT_GOAL})')
print(f'{"="*80}')
print(f'  {"Epoch":>6} | {"Rev":>7} | {"Gap":>6} | {"IC_reopt":>8} | {"IC_eval":>8} | {"IR_eval":>7} | Result')
print('  ' + '-'*70)
for r in sorted(eval_results, key=lambda x: x['epoch']):
    _flag = 'PASS ★' if r['feasible'] else 'fail  '
    print(f'  {r["epoch"]:>6} | {r["revenue_eval"]:>7.4f} | '
          f'{PROFIT_GOAL - r["revenue_eval"]:>6.4f} | '
          f'{r["regret_reopt"]:>8.4f} | '
          f'{r["regret_eval"]:>8.4f} | '
          f'{r["ir_loss_eval"]:>7.4f} | {_flag}')

## Convergence Curves with Checkpoint Markers

- **Green ★** = PASS (eval IC ≤ 0.05, revenue gap ≤ 0.1)
- **Red ✗** = fail
- Dashed green vertical = selected best checkpoint

In [ ]:
_ckpt_by_epoch = {r['epoch']: r for r in eval_results}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
_curves = [logs['reg_plus'], logs['gap_plus'], logs['ir']]
_titles = ['IC Regret+', 'Profit Gap+', 'IR Loss']
_cols   = ['tab:blue', 'tab:orange', 'tab:green']

for ax, curve, title, col in zip(axes, _curves, _titles, _cols):
    ax.plot(curve, color=col, alpha=0.85, lw=1.2)
    ax.axhline(0, color='k', lw=0.6, linestyle='--', alpha=0.4)
    ax.set_xlabel('Epoch'); ax.set_title(title)

axes[0].axhline(cfg['ic_thresh'], color='red', lw=0.8, linestyle=':', label=f'IC thresh={cfg["ic_thresh"]}')
axes[0].legend(fontsize=8)

for epoch, r in sorted(_ckpt_by_epoch.items()):
    _pass = r['feasible']
    _col  = 'limegreen' if _pass else 'crimson'
    _mk   = '*' if _pass else 'X'
    _ms   = 180 if _pass else 80
    for i, (ax, curve) in enumerate(zip(axes, _curves)):
        _y = curve[min(epoch, len(curve) - 1)]
        ax.scatter([epoch], [_y], color=_col, marker=_mk, s=_ms, zorder=6,
                   edgecolors='black' if _pass else _col, linewidths=0.5)

_legend_elems = [
    Line2D([0],[0], marker='*', color='w', markerfacecolor='limegreen',
           markeredgecolor='black', markersize=11, label='PASS (eval IC ≤ thresh)'),
    Line2D([0],[0], marker='X', color='w', markerfacecolor='crimson',
           markersize=9, label='fail'),
]
axes[2].legend(handles=_legend_elems, loc='upper right', fontsize=8)

if feasible:
    for ax in axes:
        ax.axvline(best['epoch'], color='limegreen', lw=1.5, linestyle='--', alpha=0.6)

plt.suptitle(f'{EXPERIMENT} — Training Curves (profit_goal={PROFIT_GOAL})', y=1.02)
plt.tight_layout(); plt.show()

print(f'Oracle regret (last inner step): {logs["oracle_reg_last"][-1]:.4f}')

## Final Evaluation on Best Checkpoint

In [ ]:
final_eval_keys = jax.random.split(jax.random.PRNGKey(88888), cfg['eval_num_keys'])
final_results = eval_mechanism_multikey(
    auction=auction,
    mechanism_network=mechanism_network,
    mechanism_params=best_mechanism_params,
    profit_goal=PROFIT_GOAL,
    num_samples=cfg['eval_num_samples'],
    keys=final_eval_keys,
    policy_network=policy_network,
    policy_params=best_policy_params,
    is_discrete=IS_DISCRETE,
)
print_eval_report(final_results, prefix=f'{EXPERIMENT} best checkpoint (epoch {best["epoch"]}): ')

## Save Outputs

In [ ]:
save_dir = f'outputs/{EXPERIMENT}/acceptance/seed{SEED}'
os.makedirs(save_dir, exist_ok=True)

with open(f'{save_dir}/mechanism_params.pkl', 'wb') as f:
    pickle.dump(best_mechanism_params, f)
with open(f'{save_dir}/policy_params.pkl', 'wb') as f:
    pickle.dump(best_policy_params, f)

_summary = {
    'experiment':   EXPERIMENT,
    'profit_goal':  PROFIT_GOAL,
    'best_epoch':   int(best['epoch']),
    'revenue_eval': float(best['revenue_eval']),
    'regret_reopt': float(best['regret_reopt']),
    'regret_eval':  float(best['regret_eval']),
    'ir_loss_eval': float(best['ir_loss_eval']),
    'feasible':     bool(best['feasible']),
    'num_pass':     len(feasible),
    'num_candidates': len(eval_results),
    'cfg':          {k: v for k, v in cfg.items() if not callable(v)},
    'final_eval':   {k: v for k, v in final_results.items() if isinstance(v, dict)},
}
with open(f'{save_dir}/summary.json', 'w') as f:
    json.dump(_summary, f, indent=2)

_curves_out = {
    'reg_plus': [float(x) for x in logs['reg_plus']],
    'gap_plus': [float(x) for x in logs['gap_plus']],
    'ir':       [float(x) for x in logs['ir']],
    'reg':      [float(x) for x in logs['reg']],
    'checkpoints_eval': [
        {'epoch': r['epoch'], 'revenue_eval': r['revenue_eval'],
         'regret_reopt': r['regret_reopt'], 'regret_eval': r['regret_eval'],
         'ir_loss_eval': r['ir_loss_eval'], 'feasible': r['feasible']}
        for r in sorted(eval_results, key=lambda x: x['epoch'])
    ],
}
with open(f'{save_dir}/training_curves.json', 'w') as f:
    json.dump(_curves_out, f, indent=2)

print(f'Saved to: {save_dir}/')
print(f'  mechanism_params.pkl, policy_params.pkl, summary.json, training_curves.json')